In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_excel("suez_canal_shipments_RAW_before_ETL.xlsx")

df_raw.head()


,shipment_id,type_of_vessel,type_of_cargo,shipment_date,shipment_delay_days,shipment_cost_usd,shipment_revenue_usd,cargo_net_ton,region_of_shipment,region_of_destination
0,SUEZ-2015-CNT-335739,Container Ship,Refined Oil,2015-11-06,1,970346,1546761,50841,Asia,North Africa
1,SUEZ-2015-BLK-398627,Bulk Carrier,Crude Oil,2015-09-04,14,1834339,2165401,111271,Europe,Asia
2,SUEZ-2015-BLK-350479,Bulk Carrier,Gas,2015-07-16,3,980015,882945,115856,Asia,East Asia
3,SUEZ-2015-BLK-505344,Bulk Carrier,Containers,2015-03-15,10,1631110,1555130,59489,Middle East,Europe
4,SUEZ-2015-BLK-172147,Bulk Carrier,Grain,2015-03-09,5,2053036,2444999,56063,Middle East,East Asia


In [2]:
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)


In [3]:
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"[\n\t]", "", regex=True)
        .str.replace(r"\*+|\?+", "", regex=True)
        .replace("nan", np.nan)
    )


In [4]:
date_cols = [c for c in df.columns if "date" in c]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].mode()[0])


In [5]:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = pd.to_numeric(df[col], errors="ignore")


C:\Users\user\AppData\Local\Temp\ipykernel_17012\793982244.py:3: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


In [6]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include="object").columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna("Unknown")


In [7]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)


In [8]:
df = df[~df.apply(lambda row: row.astype(str).str.lower().eq('unknown').any(), axis=1)]

In [9]:
df = df.drop_duplicates()


In [10]:
output_file = "suez_canal_shipments_cleaned.xlsx"
df.to_excel(output_file, index=False, engine='openpyxl')

print(f"Dataset successfully cleaned and saved to {output_file}")

Dataset successfully cleaned and saved to suez_canal_shipments_cleaned.xlsx
